# Modelagem — uplift de cupons iFood

Implementa `specification/02-modeling/spec.md` a partir do dataset processado
(`data/processed/`, escrito por `python -m src.pipeline`). Toda lógica vive em
`src/` — este notebook importa, executa e exibe: split temporal → baseline
preditivo → X-learner → avaliação Qini/AUUC → teste de placebo → política
sensível a custo.

**Pré-requisito:** `data/processed/` precisa existir (`uv run python -m src.pipeline`).

## 0. Setup

Raiz do repo por ancestralidade de `config.yaml`. Lê o dataset processado direto —
nenhuma transformação de pipeline roda aqui.

In [1]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

import numpy as np
import pandas as pd

from src import model_baseline, policy, split, uplift, uplift_eval
from src.config import load
from src.pipeline import build_spark

pd.set_option("display.max_columns", None)

cfg = load()
spark = build_spark(cfg, app_name="ifood-uplift-modeling")
processed = spark.read.parquet(str(cfg.processed_dir))
print(f"{processed.count():,} linhas no dataset processado.")

/home/caioolubini/Projects/ifood-coupons-uplift/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/10 00:01:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


76,277 linhas no dataset processado.


## 1. Split temporal

Treino = ondas `[0, cfg.validation_wave_cutoff)`; holdout = ondas
`[cfg.validation_wave_cutoff, cfg.n_campaign_waves)`. Nunca split aleatório.
`assert_temporal_order` rejeita qualquer corte que inverta a ordem no tempo. O mesmo
cliente pode aparecer nos dois lados (recebeu ofertas em ondas diferentes) — é
esperado, não é vazamento.

In [2]:
train_sdf, holdout_sdf = split.temporal_split(processed, cfg)
split.assert_temporal_order(train_sdf, holdout_sdf)

train_df = train_sdf.toPandas()
holdout_df = holdout_sdf.toPandas()

resumo_split = pd.DataFrame([
    {"lado": "treino", "ondas": f"[0, {cfg.validation_wave_cutoff})",
     "linhas": len(train_df), "taxa_conversao": train_df["converted"].mean()},
    {"lado": "holdout", "ondas": f"[{cfg.validation_wave_cutoff}, {cfg.n_campaign_waves})",
     "linhas": len(holdout_df), "taxa_conversao": holdout_df["converted"].mean()},
])
resumo_split

26/07/10 00:01:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,lado,ondas,linhas,taxa_conversao
0,treino,"[0, 4)",50808,0.469611
1,holdout,"[4, 6)",25469,0.398288


## 2. Baseline preditivo (logística + LGBM)

Não é o modelo de uplift: é o degrau que verifica se há sinal aprendível em
`converted` antes de pagar a complexidade do X-learner (Premissa 5). AUC é a métrica
certa aqui porque a pergunta é "dá para prever completion?". A seleção do modelo de
uplift (§3–4) não usa AUC.

In [3]:
logit, lgbm, baseline_metrics = model_baseline.train_and_log(train_df, holdout_df, cfg)
pd.Series(baseline_metrics, name="AUC (holdout)").to_frame()

,AUC (holdout)
auc_logit,0.727722
auc_lgbm,0.829110


`auc_lgbm >= auc_logit` é o aceite de T-203. Run registrado em MLflow
(`cfg.mlflow_tracking_uri`, experimento `cfg.mlflow_experiment_name`).

## 3. X-learner de uplift

Um X-learner por `offer_type` — cada tipo tem mecanismo de recompensa distinto, então
cada um aprende sua própria função de efeito. `treatment` = viu a oferta; `converted`
é o outcome. Treina no lado de treino do split.

O X-learner é τ(x) = p·τ_c(x) + (1−p)·τ_t(x), erguido sobre dois modelos de resultado:
μ₀ (ajustado só no controle) e μ₁ (só no tratado). As seções 3.2 e 3.3 abrem a caixa e
mostram esses estágios, porque um uplift alto demais costuma ser μ₀ degenerado.

### 3.1 Ajuste e uplift previsto no holdout

`uplift.predict` devolve o grão do contrato `(account_id, offer_id, received_time)` na
ordem de linha da entrada — por isso a coluna é atribuída direto, sem join. Juntar por
`(account_id, offer_id, offer_type)` seria produto cartesiano: a mesma oferta chega ao
mesmo cliente em ondas diferentes.

In [4]:
models = uplift.fit_xlearner(train_df, cfg)
uplift_holdout = uplift.predict(models, holdout_df)

holdout_df["uplift"] = uplift_holdout["uplift"].to_numpy()

por_tipo = (
    uplift_holdout.groupby("offer_type")["uplift"]
    .agg(["mean", "std", "min", "max", "count"])
    .rename(columns={"mean": "uplift_medio", "std": "desvio",
                     "min": "minimo", "max": "maximo", "count": "n"})
)
por_tipo

,uplift_medio,desvio,minimo,maximo,n
offer_type,,,,,
bogo,0.04701,0.130851,-0.480431,0.580930,10206
discount,0.06364,0.108611,-0.355721,0.729165,10206
informational,0.10209,0.166052,-0.590359,0.764735,5057


### 3.2 O outcome dentro de cada braço

O X-learner precisa que o outcome seja observável nos dois braços: se o controle não
pudesse ter `converted=1`, μ₀ não teria nenhum exemplo positivo do qual aprender.

Sob a garantia **G3** atual (`specification/schema-processed.md`), `converted` mede
compra na validade atingindo o `min_value`, **independentemente do view** — ver é o
tratamento, não o rótulo.

In [5]:
uplift.label_by_arm(holdout_df)

,offer_type,braco,n,outcome_positivo,taxa_outcome
0,bogo,controle (não viu),2366,859,0.363060
1,bogo,tratado (viu),7840,3221,0.410842
2,discount,controle (não viu),3827,808,0.211131
3,discount,tratado (viu),6379,2792,0.437686
4,informational,controle (não viu),1793,780,0.435025
5,informational,tratado (viu),3264,1684,0.515931


### 3.3 Os estágios do X-learner: μ₀, μ₁ e τ

`mu0_*` vem de `model_mu_c` (ajustado só no controle) e `mu1_*` de `models_mu_t` (só no
tratado). A identidade `τ ≈ μ₁ − μ₀` deve valer; μ₀ constante zero denunciaria um label
impossível no controle.

In [6]:
uplift.stage_diagnostics(models, holdout_df)

,offer_type,mu0_medio,mu0_desvio,mu1_medio,mu1_desvio,mu1_menos_mu0,tau_medio
0,bogo,0.368070,0.262420,0.405870,0.282651,0.037800,0.04701
1,discount,0.359035,0.314121,0.424296,0.341119,0.065261,0.06364
2,informational,0.438934,0.236164,0.521562,0.272228,0.082628,0.10209


### 3.4 Premissa causal em aberto

`treatment` = viu a oferta é **escolha do cliente**, não braço aleatorizado — o que foi
randomizado é o *envio* (Premissa 4). Quem abre a oferta tende a já ser mais ativo, e
essa auto-seleção entra na estimativa.

O uplift abaixo é causal **sob ignorabilidade condicional às features** `hist_*` e de
perfil, não por desenho experimental. A ressalva está registrada em
`specification/02-modeling/spec.md` REQ-202 e vale para tudo o que vem depois.

## 4. Avaliação Qini/AUUC

Métrica de uplift, não de classificação (REQ-203): um modelo pode acertar `converted` e
ainda ordenar mal o efeito incremental. Calculado no holdout, nunca no treino, sobre as
mesmas linhas do dataset (sem inflar por join).

In [7]:
qini_score = uplift_eval.qini(holdout_df["converted"], holdout_df["uplift"], holdout_df["treatment"])
curva = uplift_eval.qini_curve_points(holdout_df["converted"], holdout_df["uplift"], holdout_df["treatment"])
print(f"Linhas na avaliação: {len(holdout_df):,}")
print(f"Qini AUC (holdout): {qini_score:.4f}")

Linhas na avaliação: 25,469
Qini AUC (holdout): 0.0385


In [8]:
uplift_eval.fig_qini_curve(curva, qini_score, cfg, theme="light")

## 5. Teste de placebo por permutação

O Qini de 0,04 da seção anterior parece baixo — a pergunta é se ele é diferente de
zero, ou é isso que ruído produz num modelo sem nenhum efeito causal (REQ-212).

`uplift_eval.placebo_qini_distribution` embaralha `treatment` **dentro de cada
`offer_type`**, preservando a proporção tratado/controle do grupo, e refita o
X-learner do zero em cada réplica. Embaralhar globalmente mudaria essa proporção
por tipo e derrubaria o Qini nulo por composição de grupo — não pela ausência de
efeito, que é o que este teste isola. `X` e `converted` ficam fixos; só o rótulo
de tratamento embaralha.

In [9]:
qini_nulo = uplift_eval.placebo_qini_distribution(train_df, holdout_df, cfg)
resultado_placebo = uplift_eval.placebo_test(qini_score, qini_nulo, cfg)

pd.Series(resultado_placebo, name="placebo").to_frame()

,placebo
qini_real,0.038457
limiar_percentil,0.01619
passou,True
p_value,0.0
null_mean,0.000085
null_std,0.011184


`p_value` é a fração de réplicas nulas que igualam ou superam o Qini real — o
p-valor empírico da mesma distribuição, sem cálculo extra. A mesma dispersão
(`null_std`) é o intervalo de confiança do Qini reportado em §4: um cálculo,
duas leituras — significância (o Qini real fura o percentil) e incerteza (a
dispersão da nula em torno de zero).

In [10]:
uplift_eval.fig_placebo_distribution(qini_nulo, qini_score, cfg, theme="light")

## 6. Política sensível a custo (T-206) e baselines (T-207)

`policy.allocate` escolhe, por cliente, `argmax(uplift_receita − custo_esperado)` entre
as ofertas recebidas e a ação nula `nao_enviar`, cujo lucro é exatamente zero (REQ-204).

A economia do lucro é assimétrica de propósito:

- **receita é incremental** — `uplift × receita_por_conversão`, só a conversão que a
  oferta causou;
- **custo é total** — `P(converte | tratado) × discount_value`, porque o desconto é
  debitado em toda conversão da oferta, causada ou não.

`policy.offer_economics` mede `receita_por_conversão` e `discount_value` por `offer_id`
no conjunto de treino (nunca no holdout, que é a base de avaliação).

In [11]:
economics = policy.offer_economics(train_df)
economics.sort_values("discount_value")

,offer_id,offer_type,revenue_per_conversion,discount_value
0,3f207df678b143eea3cee63160fa8bed,informational,18.954841,0.0
6,5a8bc65990b245e5a138643cd4eb9837,informational,17.899284,0.0
5,fafdcd668e3743c1bb461111dcafc2a4,discount,74.688022,2.0
7,2906b810c7d4411798c6938adc9daaa5,discount,45.659014,2.0
1,2298d6c36e964ae4a3e7e9706d1fb8c2,discount,48.371701,3.0
3,f19421c1d4aa40978ebb69ca19b0e20d,bogo,36.927284,5.0
4,9b98b8c7a33c4b65b9aebfe6a799e6d9,bogo,37.716646,5.0
8,0b1e1539f2cc45b7b9fa7c272da2e1d7,discount,59.294987,5.0
2,4d5c57ea9a6940dd891ad53e9dbe8da0,bogo,44.464922,10.0
9,ae264e3637204a6fb9bb56bc8210ddfd,bogo,51.272453,10.0


`P(converte | tratado)` vem do baseline preditivo (§2) — é μ₁, não o uplift. É o LGBM
que estima a probabilidade de conversão usada no termo de custo.

In [12]:
p_convert_treated = model_baseline.predict_conversion_probability(lgbm, holdout_df)

scored = policy.expected_net_profit(
    holdout_df[["account_id", "offer_id", "offer_type", "uplift"]],
    economics,
    p_convert_treated,
)
scored[["offer_type", "uplift", "expected_revenue", "expected_cost", "net_profit"]].describe().round(4)

,uplift,expected_revenue,expected_cost,net_profit
count,25469.0000,25469.0000,25469.0000,25469.0000
mean,0.0646,2.5883,1.5757,1.0126
std,0.1322,5.4637,1.8435,5.5055
min,-0.5904,-23.5769,0.0000,-26.0262
25%,-0.0178,-0.7425,0.1205,-2.1460
50%,0.0588,2.3080,0.9960,1.0488
75%,0.1458,5.6725,2.3719,4.2335
max,0.7647,43.2358,9.6456,41.0655


### 6.1 A decisão por cliente

`allocate` devolve uma linha por cliente. `validate_recommendations` valida a saída no
contrato tipado (Pydantic nas bordas, Premissa 7).

In [13]:
decisao = policy.allocate(scored)
policy.validate_recommendations(decisao)

n_clientes = len(decisao)
n_nao_enviar = int((decisao["chosen_action"] == policy.NO_SEND).sum())

resumo_politica = pd.DataFrame([
    {"medida": "clientes no holdout", "valor": n_clientes},
    {"medida": "recebem oferta", "valor": n_clientes - n_nao_enviar},
    {"medida": f"recebem {policy.NO_SEND!r}", "valor": n_nao_enviar},
    {"medida": "% não enviar", "valor": round(100 * n_nao_enviar / n_clientes, 1)},
])
resumo_politica

,medida,valor
0,clientes no holdout,15919.0
1,recebem oferta,11283.0
2,recebem 'nao_enviar',4636.0
3,% não enviar,29.1


In [14]:
decisao["chosen_action"].value_counts().rename("clientes").to_frame()

,clientes
chosen_action,
nao_enviar,4636
2906b810c7d4411798c6938adc9daaa5,1600
5a8bc65990b245e5a138643cd4eb9837,1525
fafdcd668e3743c1bb461111dcafc2a4,1467
0b1e1539f2cc45b7b9fa7c272da2e1d7,1250
3f207df678b143eea3cee63160fa8bed,1143
2298d6c36e964ae4a3e7e9706d1fb8c2,1088
f19421c1d4aa40978ebb69ca19b0e20d,839
9b98b8c7a33c4b65b9aebfe6a799e6d9,829


### 6.2 Comparação com os baselines (REQ-205)

Três políticas de comparação, no mesmo contrato de saída:

- **aleatória** — uma oferta uniforme entre as que o cliente recebeu;
- **enviar-a-todos** — o status quo que gerou os dados; nunca escolhe a ação nula, então
  carrega lucro negativo quando toda oferta do cliente dá prejuízo;
- **top-completion** — aloca por `P(converter)` prevista, ignorando se a oferta causou a
  conversão. É o baseline que a política de uplift precisa bater.

O lucro esperado abaixo é o **do modelo**, não uma avaliação offline: a estimativa
não-enviesada por IPW sobre o holdout é T-208, ainda não implementada. Aleatória e
top-completion não carregam lucro esperado próprio (escolhem sem otimizar lucro), por
isso aparecem só com a ação escolhida.

In [15]:
send_all = policy.policy_send_all(scored)
top_completion = policy.policy_top_completion(holdout_df, p_convert_treated)
aleatoria = policy.policy_random(holdout_df, cfg)

comparacao = pd.DataFrame([
    {"politica": "uplift (T-206)", "lucro_esperado_por_cliente": decisao["expected_net_profit"].mean(),
     "clientes_sem_envio": n_nao_enviar},
    {"politica": "enviar-a-todos", "lucro_esperado_por_cliente": send_all["expected_net_profit"].mean(),
     "clientes_sem_envio": 0},
    {"politica": "top-completion", "lucro_esperado_por_cliente": np.nan, "clientes_sem_envio": 0},
    {"politica": "aleatória", "lucro_esperado_por_cliente": np.nan, "clientes_sem_envio": 0},
])
comparacao.round(4)

,politica,lucro_esperado_por_cliente,clientes_sem_envio
0,uplift (T-206),3.5040,4636
1,enviar-a-todos,2.5216,0
2,top-completion,NaN,0
3,aleatória,NaN,0


In [16]:
prejuizo = send_all[send_all["expected_net_profit"] < 0]
print(f"Clientes a quem 'enviar-a-todos' manda oferta com lucro esperado negativo: {len(prejuizo):,}")
print(f"Prejuízo esperado agregado desses envios: R$ {prejuizo['expected_net_profit'].sum():,.2f}")
print(f"Ganho da política sobre o status quo:     R$ "
      f"{decisao['expected_net_profit'].sum() - send_all['expected_net_profit'].sum():,.2f}")

Clientes a quem 'enviar-a-todos' manda oferta com lucro esperado negativo: 4,636
Prejuízo esperado agregado desses envios: R$ -15,638.30
Ganho da política sobre o status quo:     R$ 15,638.30


## Próximo passo

`specification/02-modeling/tasks.md`:

- **T-208** — avaliação offline por IPW com propensity fixa, checando positividade antes
  de reportar qualquer valor (REQ-206, REQ-207). É o que transforma o lucro esperado
  acima numa estimativa não-enviesada sobre o holdout.
- **T-209** — impacto em R$ com intervalo e dimensionamento do próximo A/B.
- **T-210** — SHAP global e force plot individual.
- **T-211** — consolidação deste notebook para o slide.

In [17]:
spark.stop()